In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sn

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df.shape

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
target = 'is_fraud'

In [ ]:
print(train_df[target].value_counts())
print(train_df[target].mean())

In [ ]:
sn.countplot(x=train_df[target])
plt.title('is_fraud distribution')
plt.savefig('is_fraud distribution.png')
plt.show()

In [ ]:
bool_cols = train_df.select_dtypes(include=['bool']).columns.tolist()
train_df[bool_cols] = train_df[bool_cols].astype(int)
test_df[bool_cols] = test_df[bool_cols].astype(int)
bool_cols

In [ ]:
x = train_df.drop(columns=[target, 'id'])
y = train_df[target]

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print(len(x_train), len(x_val), len(y_train), len(y_val))

In [ ]:
numerical_features = x_train.select_dtypes(include=['number']).columns.tolist()

In [ ]:
nominal_features = x_train.select_dtypes(exclude=['number']).columns.tolist()

In [ ]:
print(numerical_features)
print(nominal_features)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
num_trans = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
nom_trans = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("numerical_features", num_trans, numerical_features),
    ("nominal_features", nom_trans, nominal_features)
])

In [ ]:
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model_classifier", HistGradientBoostingClassifier(
        max_iter=500, learning_rate=0.05, max_depth=6,
        early_stopping=True, random_state=42))
])

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:

clf.fit(x_train, y_train)
val_pred = clf.predict_proba(x_val)[:, 1]
print("Validation ROC AUC:", roc_auc_score(y_val, val_pred))

In [ ]:
clf.fit(x, y)

In [ ]:
test_pred = clf.predict_proba(test_df.drop(columns=['id']))[:, 1]

In [ ]:
submission = pd.DataFrame({"id": test_df["id"], "is_fraud": test_pred})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()